In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import random
import re
from collections import Counter

from skipgram import SkipGram

In [2]:
torch.manual_seed(1337)

In [3]:
with open("./dataset/emma.txt", "r", encoding="utf-8") as file:
    corpus = file.read().splitlines()

In [4]:
# Tokenization - Ignore casing and special characters
def tokenize(line):
    return re.findall(r"[a-z']+", line.lower())
    
tokens = []
for line in corpus:
    tokens.extend(tokenize(line.strip()))

word_counts = Counter(tokens)

In [5]:
vocab = sorted(word_counts.keys())
vocab_size = len(vocab)
word_to_idx, idx_to_word = {}, {}

for i, word in enumerate(vocab):
    word_to_idx[word] = i
    idx_to_word[i] = word

In [6]:
# Mikolov subsampling
total_words = sum(word_counts.values())
subsampling_threshold = 1e-3

keep_probs = {}
def get_keep_probability(count):
    return min(1.0, (subsampling_threshold / (count / total_words)) ** 0.5)
    
for w, c in word_counts.items():
    keep_probs[w] = get_keep_probability(c)

In [7]:
training_pairs = []
window_size = 5

for line in corpus:
    words = [w for w in tokenize(line) if random.random() < keep_probs[w]]

    for i, center_word in enumerate(words):
        start = max(0, i-window_size)
        end = min(i+window_size, len(words)-1)

        for j in range(start, end+1):
            if i == j:
                continue

            pair = (word_to_idx[center_word], word_to_idx[words[j]])
            training_pairs.append(pair)
    
training_pairs = torch.tensor(
    training_pairs,
    dtype=torch.long
)

In [8]:
frequencies = torch.tensor([word_counts[word] for word in vocab], dtype=torch.float)

neg_sampling_dist = frequencies ** 0.75
neg_sampling_dist /= neg_sampling_dist.sum()

def get_negative_samples(batch_size, n_negatives):
    samples = torch.multinomial(neg_sampling_dist, num_samples=batch_size*n_negatives, replacement=True)
    return samples.view(batch_size, n_negatives)

In [9]:
def loss_fn(p_scores, n_scores):
    p_loss = -F.logsigmoid(p_scores)
    n_loss = -F.logsigmoid(-n_scores).sum(dim=1)

    loss = (p_loss + n_loss).mean()
    return loss

In [10]:
embedding_dim = 100
epochs = 40
lr = 1e-2

batch_size = 1024
n_negatives = 5
n_pairs = len(training_pairs)

model = SkipGram(vocab_size, embedding_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [11]:
# Training
model.train()
for epoch in range(epochs):
    indices = torch.randperm(n_pairs)
    total_loss, count = 0, 0

    for start_idx in range(0, n_pairs, batch_size):
        batch_indices = indices[start_idx: start_idx + batch_size]
        batch = training_pairs[batch_indices]

        center_ids = batch[:, 0]
        positive_ids = batch[:, 1]
        negative_ids = get_negative_samples(len(batch), n_negatives)

        optimizer.zero_grad()

        p_scores, n_scores = model(center_ids, positive_ids, negative_ids)
        loss = loss_fn(p_scores, n_scores)

        total_loss += loss.item() * len(batch)
        count += len(batch)

        loss.backward()
        optimizer.step()

    average_loss = total_loss / count

    print(f"Epoch: {epoch + 1}, Loss: {average_loss}")

print(f"Final Loss: {average_loss}")

Epoch: 1, Loss: 2.7431669787972868
Epoch: 2, Loss: 2.4631279379137334
Epoch: 3, Loss: 2.252776365713746
Epoch: 4, Loss: 2.158248451713239
Epoch: 5, Loss: 2.1072107787248275
Epoch: 6, Loss: 2.079913661221543
Epoch: 7, Loss: 2.0598938729015486
Epoch: 8, Loss: 2.0471990199800354
Epoch: 9, Loss: 2.0351180486872646
Epoch: 10, Loss: 2.0277472044018015
Epoch: 11, Loss: 2.0223547132232857
Epoch: 12, Loss: 2.0172710115087833
Epoch: 13, Loss: 2.012636671738884
Epoch: 14, Loss: 2.008722984064015
Epoch: 15, Loss: 2.0057699900558132
Epoch: 16, Loss: 2.0038055470345477
Epoch: 17, Loss: 2.00030881159729
Epoch: 18, Loss: 2.001219253207295
Epoch: 19, Loss: 1.996906429294288
Epoch: 20, Loss: 1.9953189139530827
Epoch: 21, Loss: 1.9952636558639545
Epoch: 22, Loss: 1.9935463540404161
Epoch: 23, Loss: 1.9959098622138822
Epoch: 24, Loss: 1.9902719973130016
Epoch: 25, Loss: 1.989644023141654
Epoch: 26, Loss: 1.9909860858062098
Epoch: 27, Loss: 1.9907021671180152
Epoch: 28, Loss: 1.9876783712972743
Epoch: 29, 

In [12]:
word_embeddings = model.W_in.detach()
print(word_embeddings.shape)

torch.Size([7224, 100])


In [13]:
# L2 Normalization
norm_vectors = F.normalize(word_embeddings, p=2, dim=1)

def most_similar(word, top_k=5):
    if word not in word_to_idx:
        raise KeyError(f"'{word}' not in vocabulary")

    word_id = word_to_idx[word]

    sims = norm_vectors @ norm_vectors[word_id]
    sims[word_id] = -1

    scores, indices = torch.topk(sims, k=top_k)
    return [(idx_to_word[i.item()], s.item()) for i, s in zip(indices, scores)]

most_similar("pleasant")

[('genial', 0.4661731719970703),
 ('elegantly', 0.45344704389572144),
 ('handled', 0.44669264554977417),
 ('sweetly', 0.4317495822906494),
 ('fill', 0.42875444889068604)]